# វិភាគការទាមទារចំណាយ

កំណត់ត្រានេះបង្ហាញពីវិធីបង្កើតភ្នាក់ងារដែលប្រើកម្មវិធីបន្ថែមដើម្បីដំណើរការចំណាយធម្មតារប្រទេសពីរូបភាពប័ណ្ណបញ្ជាទិញក្នុងស្រុក បង្កើតអ៊ីមែលទាមទារចំណាយ និងបង្ហាញទិន្នន័យចំណាយដោយប្រើតារាងផ្លែម៉άν។ ភ្នាក់ងារជ្រើសរើសមុខងារដោយ δυναμικά ដោយផ្អែកលើបរិបទភារកិច្ច។

ជំហានៈ
1. ភ្នាក់ងារ OCR ដំណើរការរូបភាពប័ណ្ណបញ្ជាទិញនៅក្នុងស្រុក និងផ្ដល់ព័ត៌មានចំណាយធម្មតារប្រទេស។
2. ភ្នាក់ងារ Email បង្កើតអ៊ីមែលទាមទារចំណាយ។

### ឧទាហរណ៍ស្ថានភាពចំណាយធម្មតារប្រទេស៖
សូមគំនិតថាអ្នកជាអ្នកបម្រើការធ្វើដំណើរព្រមព្រៀងអាជីវកម្មនៅទីក្រុងផ្សេង ។ ក្រុមហ៊ុនរបស់អ្នកមានគោលការណ៍ដែលសងប្រាក់វិញចំពោះចំណាយទូទៅទាក់ទងនឹងការធ្វើដំណើរទាំងអស់ដែលសមរម្យ។ នេះជាការបំបែកចំណាយធម្មតារប្រទេសដែលអាចមាន៖
- ការដឹកជញ្ជូន៖
សំបុត្រយន្តហោះទៅវិញទៅមកពីទីក្រុងផ្ទះរបស់អ្នកទៅទីក្រុងគោលដៅ។
តាក់ស៊ីឬសេវាកម្មហៅឡានទៅនិងមកតាមព្រលានយន្តហោះ។
ការដឹកជញ្ជូនក្នុងស្រុកនៅទីក្រុងគោលដៅ (ដូចជារថយន្តសាធារណៈ រថយន្តជួល ឬតាក់ស៊ី)។

- ការស្នាក់នៅ៖
សណ្ឋាគារស្នាក់នៅបីយប់នៅសណ្ឋាគារជំនាន់មធ្យមនៅជិតកន្លែងប្រជុំ។

- អាហារ៖
ប្រាក់ឲ្យសំរាប់អាហារប្រចាំថ្ងៃសម្រាប់អាហារពេលព្រឹក អាហារថ្ងៃ និងអាហារពេលល្ងាច ដោយផ្អែកលើគោលការណ៍ប្រាក់អាហារប្រាក់ចំណាយថ្ងៃ។

- ចំណាយផ្សេងទៀត៖
ថ្លៃចតឡាននៅព្រលានយន្តហោះ។
ថ្លៃចូលប្រើអ៊ីនធឺណិតនៅសណ្ឋាគារ។
កាដូឬថ្លៃសេវាកម្មតូចៗ។

- ឯកសារ៖
អ្នកដាក់ស្នើប័ណ្ណបញ្ជាទិញទាំងអស់ (សំបុត្រយន្តហោះ តាក់ស៊ី សណ្ឋាគារ អាហារ ជាដើម) និងរបាយការណ៍ចំណាយដែលបានបំពេញសម្រាប់ការសងប្រាក់វិញ។


## នាំចូលបណ្ណាល័យដែលត្រូវការ

នាំចូលបណ្ណាល័យ និងម៉ូឌុលដែលត្រូវការសម្រាប់កំណត់ត្រា។


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## កំណត់គំរូចំណាយ

 បង្កើតគំរូ Pydantic សម្រាប់ចំណាយជារូបមន្តនិងថ្នាក់ ExpenseFormatter ផ្ទៀងផ្ទាត់សំណួររបស់អ្នកប្រើជាទិន្នន័យចំណាយដែលមានរចនាសម្ព័ន្ធ។

 ចំណាយនីមួយៗនឹងត្រូវតំណាងដោយរចនាសម្ព័ន្ធ៖
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## ការបញ្ជាក់ឧបករណ៍ - ការបង្កើតអ៊ីមែល

បង្កើតមុខងារឧបករណ៍ដើម្បីបង្កើតអ៊ីមែលសម្រាប់ដាក់ស្នើការទាមទារចំណាយ។
- ឧបករណ៍នេះប្រើ `@tool` ពី Microsoft Agent Framework។
- វាគណនាចំនួនសរុបនៃ ចំណាយ ហើយបំលែងព័ត៌មានលម្អិតទៅជារូបរាងអ៊ីមែល។


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ឧបករណ៍សម្រាប់ដកចំណាយធ្វើដំណើរពីរូបភាពវិក្កយបត្រ

បង្កើតមុខងារឧបករណ៍ដើម្បីដកចំណាយធ្វើដំណើរពីរូបភាពវិក្កយបត្រ។
- ឧបករណ៍នេះប្រើ `@tool` decorator ពី Microsoft Agent Framework។
- វាอ่านข้อความរូបភាពវិក្កយបត្រ, រួចបំលែងវាទៅជា base64, ហើយត្រលប់ URI ទិន្នន័យសម្រាប់ភ្នាក់ងារដើម្បីវិភាគ។


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ការដំណើរការចំណាយ

កំណត់ភ្នាក់ងារនិងភ្ជាប់ពួកគេទៅក្នុងដំណើរការដោយជាប់លំដោយដោយប្រើ `WorkflowBuilder`។
- ភ្នាក់ងារ OCR យកទិន្នន័យចំណាយដែលមានរចនាសម្ព័ន្ធពីរូបភាពវិក័យបត្រដោយប្រើឧបករណ៍ `load_receipt_image`។
- ភ្នាក់ងារ អ៊ីមែល យកទិន្នន័យដែលត្រូវបានយកហើយបង្កើតអ៊ីមែលប្តឹងចំណាយដែលមានវិជ្ជាជីវៈ ដោយប្រើឧបករណ៍ `generate_expense_email`។
- `WorkflowBuilder` ជាមួយ `add_edge` បង្កើតបណ្ដាញជាប់លំដាប់៖ ភ្នាក់ងារ OCR → ភ្នាក់ងារ អ៊ីមែល។


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## មុខងារសំខាន់

សង់ផ្លូវការងារតាមលំដាប់ ហើយរត់វាដើម្បីដំណើរការរូបភាពបង្កាន់ដៃ និងបង្កើតអ៊ីមែលការទាមទារចំណាយ។


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
